# Direction A: Matched-Format Poetry Replication

**Goal**: Redo Exp 2 (subspace), Exp 3 (emergence), Exp 5 (neurons) using **matched-format** data — **high-imagery poems vs low-imagery poems** — instead of poetry vs news (Exp 9) or synthetic pairs (Exp 2).

**Hypothesis**: When both groups are poems (same format), the model should need deeper layers to distinguish them, and the 15-layer emergence gap may partially return.

**Data**: 4,900 Chinese poems scored by nature/sensory imagery density. Top-50 (most imagistic) vs bottom-50 (most abstract/didactic) selected as matched pairs.

In [2]:
# §1 Setup
import os, sys, json, time, warnings
warnings.filterwarnings('ignore')
os.environ['GGML_CUDA_DISABLE_GRAPHS'] = '1'
os.environ['HF_HOME'] = '/workspace/Data/huggingface_cache'

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import roc_auc_score
from scipy.stats import ttest_ind

plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'Noto Sans CJK JP', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path('/workspace/Data/poetry_data')
OUTPUT_DIR = Path('/workspace/Data/direction_A')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
N_SAMPLES = 50  # per group
MODEL_PATH = '/workspace/Model/Qwen3-8B-GGUF/Qwen3-8B-Q4_K_M.gguf'
NOTHINK_SUFFIX = '<think>\n\n</think>\n\n'

# Load reference data
exp2_data = np.load('/workspace/Data/aesthetic_experiment/aesthetic_analysis_results.npz', allow_pickle=True)
exp3_data = np.load('/workspace/Data/aesthetic_emergence/emergence_analysis_results.npz', allow_pickle=True)
exp5_data = np.load('/workspace/Data/aesthetic_neurons/ablation_results.npz', allow_pickle=True)
exp9_data = np.load('/workspace/Data/poetry_replication/poetry_replication_results.npz', allow_pickle=True)

exp2_best = int(exp2_data['best_cls_layer'])
exp2_dir_norm = exp2_data['aesthetic_direction_norm']  # (36, 4096)
exp9_dir_norm = exp9_data['aesthetic_dir_norm']

# Aesthetic word set for generation density measurement
AESTHETIC_WORDS = {
    'beautiful', 'beauty', 'elegant', 'elegance', 'graceful', 'grace',
    'harmony', 'harmonious', 'serene', 'serenity', 'sublime', 'exquisite',
    'delicate', 'refined', 'magnificent', 'splendid', 'radiant', 'luminous',
    'tranquil', 'ethereal', 'enchanting', 'captivating', 'majestic',
    'poetic', 'lyrical', 'tender', 'gentle', 'vibrant', 'vivid'
}

print(f'Exp 2 reference: best_layer={exp2_best}')
print(f'Output dir: {OUTPUT_DIR}')

Exp 2 reference: best_layer=16
Output dir: /workspace/Data/direction_A


In [ ]:
# §2 Load Engine
sys.path.insert(0, '/workspace/NeuroScope')
import neuroscope

engine = neuroscope.Engine(MODEL_PATH, n_ctx=4096, n_seq_max=1, n_gpu_layers=99)
info = engine.model_info
N_LAYERS, N_EMBD = info.n_layers, info.n_embd
print(f'Model: {info.name} | {N_LAYERS} layers | {N_EMBD} dim | ctx={info.n_ctx}')

Model: Qwen3 8B Awq Compatible Instruct | 36 layers | 4096 dim | ctx=4096


ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GB10, compute capability 12.1, VMM: yes
ggml_backend_cuda_get_available_uma_memory: final available_memory_kb: 85150572
ggml_backend_cuda_get_available_uma_memory: final available_memory_kb: 85150572
llama_model_load_from_file_impl: using device CUDA0 (NVIDIA GB10) (000f:01:00.0) - 83154 MiB free
llama_model_loader: loaded meta data with 28 key-value pairs and 399 tensors from /workspace/Model/Qwen3-8B-GGUF/Qwen3-8B-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen3 8B Awq Compatible Instruct
llama_model_loader: - kv   3:                           general.f

er   8: dev = CUDA0
llama_kv_cache: layer   9: dev = CUDA0
llama_kv_cache: layer  10: dev = CUDA0
llama_kv_cache: layer  11: dev = CUDA0
llama_kv_cache: layer  12: dev = CUDA0
llama_kv_cache: layer  13: dev = CUDA0
llama_kv_cache: layer  14: dev = CUDA0
llama_kv_cache: layer  15: dev = CUDA0
llama_kv_cache: layer  16: dev = CUDA0
llama_kv_cache: layer  17: dev = CUDA0
llama_kv_cache: layer  18: dev = CUDA0
llama_kv_cache: layer  19: dev = CUDA0
llama_kv_cache: layer  20: dev = CUDA0
llama_kv_cache: layer  21: dev = CUDA0
llama_kv_cache: layer  22: dev = CUDA0
llama_kv_cache: layer  23: dev = CUDA0
llama_kv_cache: layer  24: dev = CUDA0
llama_kv_cache: layer  25: dev = CUDA0
llama_kv_cache: layer  26: dev = CUDA0
llama_kv_cache: layer  27: dev = CUDA0
llama_kv_cache: layer  28: dev = CUDA0
llama_kv_cache: layer  29: dev = CUDA0
llama_kv_cache: layer  30: dev = CUDA0
llama_kv_cache: layer  31: dev = CUDA0
llama_kv_cache: layer  32: dev = CUDA0
llama_kv_cache: layer  33: dev = CUDA0
llama

In [4]:
# §3 Load poems + imagery scoring → select matched high/low groups
zh_poems_all = json.load(open(DATA_DIR / 'chinese_poems.json'))
print(f'Total ZH poems: {len(zh_poems_all)}')

# Nature/sensory imagery characters
IMAGERY_CHARS = set('月花风雪春秋山水云霞星夜雨露竹松柳桃兰菊荷梅鸟蝶泉湖海虹烟雾光影梦香')

# Score each poem by imagery density
scores = []
for p in zh_poems_all:
    c = p['content']
    s = sum(1 for ch in c if ch in IMAGERY_CHARS) / max(len(c), 1)
    scores.append(s)
scores = np.array(scores)

# Select top-N and bottom-N by imagery score
# Filter: require length 20-200 chars (standard poem length)
valid_mask = np.array([20 <= len(p['content']) <= 200 for p in zh_poems_all])
valid_scores = scores.copy()
valid_scores[~valid_mask] = -1  # exclude from top
rng = np.random.RandomState(SEED)

# Top group: highest imagery density
top_candidates = np.argsort(valid_scores)[::-1][:N_SAMPLES * 3]  # pool of 150
top_idx = rng.choice(top_candidates, N_SAMPLES, replace=False)

# Bottom group: lowest imagery density (but valid length)
valid_scores2 = scores.copy()
valid_scores2[~valid_mask] = 999  # exclude from bottom
bot_candidates = np.argsort(valid_scores2)[:N_SAMPLES * 3]
bot_idx = rng.choice(bot_candidates, N_SAMPLES, replace=False)

high_poems = [zh_poems_all[i]['content'][:300] for i in top_idx]
low_poems = [zh_poems_all[i]['content'][:300] for i in bot_idx]

high_scores = scores[top_idx]
low_scores = scores[bot_idx]

print(f'High-imagery group: N={len(high_poems)}, mean score={high_scores.mean():.4f} (range {high_scores.min():.3f}-{high_scores.max():.3f})')
print(f'Low-imagery group:  N={len(low_poems)}, mean score={low_scores.mean():.4f} (range {low_scores.min():.3f}-{low_scores.max():.3f})')
print(f'Effect size (d): {(high_scores.mean()-low_scores.mean()) / np.sqrt((high_scores.std()**2+low_scores.std()**2)/2):.2f}')
print(f'\nHigh-imagery examples:')
for i in range(3):
    print(f'  [{high_scores[i]:.3f}] {high_poems[i][:60]}')
print(f'\nLow-imagery examples:')
for i in range(3):
    print(f'  [{low_scores[i]:.3f}] {low_poems[i][:60]}')

# Match length distributions
high_lens = np.array([len(p) for p in high_poems])
low_lens = np.array([len(p) for p in low_poems])
print(f'\nLength: high={high_lens.mean():.0f}±{high_lens.std():.0f}, low={low_lens.mean():.0f}±{low_lens.std():.0f}')

all_texts = high_poems + low_poems
labels = np.array([1]*N_SAMPLES + [0]*N_SAMPLES)  # 1=high-imagery, 0=low-imagery
print(f'\nTotal: {len(all_texts)} texts ({sum(labels)} high, {len(labels)-sum(labels)} low)')

Total ZH poems: 4900
High-imagery group: N=50, mean score=0.1824 (range 0.152-0.250)
Low-imagery group:  N=50, mean score=0.0000 (range 0.000-0.000)
Effect size (d): 8.58

High-imagery examples:
  [0.167] 愁见唱阳春，令人离肠结。郎去未归家，柳自飘香雪。
  [0.219] 白水茫茫是我家，船头明月照芦花。晚风吹醒江湖梦，挂起纶竿夜煮茶。
  [0.156] 错认秋光忽似春，相逢疑假梦疑真。暗中添得如云侣，翻恨缘深误作邻。

Low-imagery examples:
  [0.000] 保庇孤根逢圣主，矜修晚节顺天机。空门自有清凉地，不向红尘议是非。
  [0.000] 昏昏健忘废专精，默坐空斋忽自惊。
少作回看如两手，旧书重读似前生。
却疑安世知三箧，不晓睢阳记一城。
莫怪诗成呼烛写，晓
  [0.000] 疲於吏事老於兵，不特头方命亦屯。
成败固非予逆睹，穷通何用子前陈。
登朝不是鸢肩相，守塞尤惭燕颔人。
见说江东多福德，亟

Length: high=37±12, low=40±14

Total: 100 texts (50 high, 50 low)


---
## Part A: Aesthetic Subspace (cf. Exp 2, Exp 9)

Both groups are Chinese poems — **same language, same format**. The only difference is imagery density.

In [5]:
# §4 Collect all-layer activations
def collect_all_activations(engine, texts, desc='Collecting'):
    all_acts = []
    t0 = time.time()
    for i, text in enumerate(texts):
        messages = [{'role': 'user', 'content': text}]
        templated = engine.apply_chat_template(messages) + NOTHINK_SUFFIX
        engine.reset()
        engine.forward(templated, add_special=True)
        acts = engine.get_all_activations()
        all_acts.append(np.array(acts))
        if (i + 1) % 25 == 0:
            print(f'  {desc}: {i+1}/{len(texts)} ({time.time()-t0:.1f}s)')
    print(f'  {desc}: done in {time.time()-t0:.1f}s')
    return np.array(all_acts)

print('=== Collecting activations (100 matched poems) ===')
t_start = time.time()
acts_high = collect_all_activations(engine, high_poems, 'High-imagery')
acts_low = collect_all_activations(engine, low_poems, 'Low-imagery')
acts_all = np.concatenate([acts_high, acts_low])
print(f'Total: {time.time()-t_start:.1f}s | shapes: high={acts_high.shape}, low={acts_low.shape}')

=== Collecting activations (100 matched poems) ===
  High-imagery: 25/50 (1.3s)
  High-imagery: 50/50 (2.5s)
  High-imagery: done in 2.5s
  Low-imagery: 25/50 (1.2s)
  Low-imagery: 50/50 (2.4s)
  Low-imagery: done in 2.4s
Total: 5.0s | shapes: high=(50, 36, 4096), low=(50, 36, 4096)


In [6]:
# §5 Direction, classification, comparison with Exp 2 & 9
# Aesthetic direction: mean(high) - mean(low)
mean_high = acts_high.mean(axis=0)  # (36, 4096)
mean_low = acts_low.mean(axis=0)
aesthetic_dir = mean_high - mean_low
aesthetic_dir_norm = aesthetic_dir / (np.linalg.norm(aesthetic_dir, axis=1, keepdims=True) + 1e-10)

# Direction magnitude by layer
dir_norms = np.linalg.norm(aesthetic_dir, axis=1)

# Cosine with Exp 2 and Exp 9 directions
cos_exp2 = np.array([np.dot(aesthetic_dir_norm[l], exp2_dir_norm[l]) for l in range(N_LAYERS)])
cos_exp9 = np.array([np.dot(aesthetic_dir_norm[l], exp9_dir_norm[l]) for l in range(N_LAYERS)])

# Classification by layer (5-fold CV)
layer_aucs = np.zeros((N_LAYERS, 2))
for l in range(N_LAYERS):
    X = acts_all[:, l, :]
    clf = LogisticRegression(max_iter=2000, random_state=SEED)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    aucs = cross_val_score(clf, X, labels, cv=skf, scoring='roc_auc')
    layer_aucs[l] = [aucs.mean(), aucs.std()]

best_layer = np.argmax(layer_aucs[:, 0])
best_auc = layer_aucs[best_layer, 0]

# Also compute simple projection AUC
proj_aucs = np.zeros(N_LAYERS)
for l in range(N_LAYERS):
    projs = acts_all[:, l, :] @ aesthetic_dir_norm[l]
    proj_aucs[l] = roc_auc_score(labels, projs)

# PCA at best layer
pca = PCA(n_components=5, random_state=SEED)
X_pca = pca.fit_transform(acts_all[:, best_layer, :])

print(f'Best classification layer: L{best_layer} (AUC={best_auc:.4f})')
print(f'AUC at L0: {layer_aucs[0, 0]:.4f}')
print(f'AUC at L16 (Exp 2 best): {layer_aucs[16, 0]:.4f}')
print(f'PCA explained variance (L{best_layer}): {pca.explained_variance_ratio_[:3]}')
print(f'\nDirection cosine with Exp 2 @ L16: {cos_exp2[16]:.3f}')
print(f'Direction cosine with Exp 9 @ L0:  {cos_exp9[0]:.3f}')
print(f'Direction cosine with Exp 2 @ best: {cos_exp2[best_layer]:.3f}')

# --- Plot ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Part A: Matched-Format Subspace', fontsize=14, fontweight='bold')

# 1. Classification by layer
ax = axes[0, 0]
ax.plot(range(N_LAYERS), layer_aucs[:, 0], 'b-o', ms=3, label='Logistic (5-fold)')
ax.plot(range(N_LAYERS), proj_aucs, 'r--', alpha=0.7, label='Projection')
ax.axhline(0.95, ls=':', c='gray', alpha=0.5)
ax.axvline(best_layer, ls='--', c='orange', alpha=0.7, label=f'Best=L{best_layer}')
ax.axvline(16, ls=':', c='green', alpha=0.5, label='Exp 2 best (L16)')
ax.set_xlabel('Layer'); ax.set_ylabel('AUC'); ax.set_title('Classification by Layer')
ax.legend(fontsize=8); ax.set_ylim(0.4, 1.05)

# 2. Direction cosine comparison
ax = axes[0, 1]
ax.plot(range(N_LAYERS), cos_exp2, 'g-o', ms=3, label='vs Exp 2 (synthetic)')
ax.plot(range(N_LAYERS), cos_exp9, 'r-s', ms=3, label='vs Exp 9 (poetry-news)')
ax.set_xlabel('Layer'); ax.set_ylabel('Cosine similarity')
ax.set_title('Direction Alignment'); ax.legend(fontsize=8)
ax.axhline(0, ls=':', c='gray', alpha=0.3)

# 3. Direction magnitude
ax = axes[1, 0]
ax.plot(range(N_LAYERS), dir_norms, 'b-o', ms=3, label='Matched (high-low)')
exp2_norms = np.linalg.norm(exp2_data['aesthetic_direction'], axis=1) if 'aesthetic_direction' in exp2_data else None
if exp2_norms is not None:
    ax.plot(range(N_LAYERS), exp2_norms, 'g--', alpha=0.7, label='Exp 2')
ax.set_xlabel('Layer'); ax.set_ylabel('L2 Norm')
ax.set_title('Direction Magnitude'); ax.legend(fontsize=8)

# 4. PCA at best layer
ax = axes[1, 1]
ax.scatter(X_pca[labels==1, 0], X_pca[labels==1, 1], c='crimson', alpha=0.6, s=30, label='High-imagery')
ax.scatter(X_pca[labels==0, 0], X_pca[labels==0, 1], c='steelblue', alpha=0.6, s=30, label='Low-imagery')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'PCA @ L{best_layer}'); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'partA_subspace.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved partA_subspace.png')

Best classification layer: L4 (AUC=0.9680)
AUC at L0: 0.9520
AUC at L16 (Exp 2 best): 0.9520
PCA explained variance (L4): [0.12062292 0.08075086 0.06930713]

Direction cosine with Exp 2 @ L16: 0.166
Direction cosine with Exp 9 @ L0:  -0.265
Direction cosine with Exp 2 @ best: 0.029
Saved partA_subspace.png


In [7]:
# §6 AUC threshold analysis + Part A summary
# Find first layer where AUC >= thresholds
thresholds = [0.75, 0.90, 0.95, 0.99]
threshold_layers = {}
for t in thresholds:
    layers_above = np.where(layer_aucs[:, 0] >= t)[0]
    threshold_layers[t] = int(layers_above[0]) if len(layers_above) > 0 else -1
    print(f'AUC >= {t}: Layer {threshold_layers[t]}' + (' (never)' if threshold_layers[t] == -1 else ''))

print(f'\n{"="*60}')
print(f'PART A SUMMARY: Matched-Format Subspace')
print(f'{"="*60}')
print(f'{"Metric":<35} {"Exp 2":>10} {"Exp 9":>10} {"Dir A":>10}')
print(f'{"-"*65}')
print(f'{"Best layer":<35} {"L16":>10} {"L0":>10} {"L"+str(best_layer):>10}')
print(f'{"Peak AUC":<35} {"1.000":>10} {"1.000":>10} {best_auc:>10.4f}')
print(f'{"AUC @ L0":<35} {"0.938":>10} {"1.000":>10} {layer_aucs[0,0]:>10.4f}')
print(f'{"AUC @ L16":<35} {"1.000":>10} {"1.000":>10} {layer_aucs[16,0]:>10.4f}')
print(f'{"Cosine vs Exp 2 @ L16":<35} {"1.000":>10} {"0.310":>10} {cos_exp2[16]:>10.3f}')
print(f'{"PCA PC1 variance":<35} {"34.0%":>10} {"38.2%":>10} {pca.explained_variance_ratio_[0]:>10.1%}')

AUC >= 0.75: Layer 0
AUC >= 0.9: Layer 0
AUC >= 0.95: Layer 0
AUC >= 0.99: Layer -1 (never)

PART A SUMMARY: Matched-Format Subspace
Metric                                   Exp 2      Exp 9      Dir A
-----------------------------------------------------------------
Best layer                                 L16         L0         L4
Peak AUC                                 1.000      1.000     0.9680
AUC @ L0                                 0.938      1.000     0.9520
AUC @ L16                                1.000      1.000     0.9520
Cosine vs Exp 2 @ L16                    1.000      0.310      0.166
PCA PC1 variance                         34.0%      38.2%      12.1%


---
## Part B: Aesthetic Emergence (cf. Exp 3)

Does the **perception–judgment gap** return with matched-format data?

In [8]:
# §7 Judgment probing — wrap each poem in evaluation template
JUDGMENT_TEMPLATE = '请仔细阅读以下诗歌，然后评价：这首诗的意境是否优美？\n\n{text}\n\n你的评价：'

print('=== Collecting judgment-probing activations ===')
judgment_acts = []
t0 = time.time()
for i, text in enumerate(all_texts):
    judgment_text = JUDGMENT_TEMPLATE.format(text=text)
    msgs = [{'role': 'user', 'content': judgment_text}]
    templated = engine.apply_chat_template(msgs) + NOTHINK_SUFFIX
    engine.reset()
    engine.forward(templated, add_special=True)
    acts = engine.get_all_activations()
    judgment_acts.append(np.array(acts))
    if (i + 1) % 25 == 0:
        print(f'  Judgment: {i+1}/{len(all_texts)} ({time.time()-t0:.1f}s)')
judgment_acts = np.array(judgment_acts)
print(f'  Done in {time.time()-t0:.1f}s | shape={judgment_acts.shape}')

=== Collecting judgment-probing activations ===
  Judgment: 25/100 (1.3s)
  Judgment: 50/100 (2.6s)
  Judgment: 75/100 (3.9s)
  Judgment: 100/100 (5.2s)
  Done in 5.2s | shape=(100, 36, 4096)


In [9]:
# §8 Emergence curves: content probing vs judgment probing
# Content probing (from §5 acts_all)
content_aucs = layer_aucs.copy()  # already computed

# Judgment probing
judgment_aucs = np.zeros((N_LAYERS, 2))
for l in range(N_LAYERS):
    X = judgment_acts[:, l, :]
    clf = LogisticRegression(max_iter=2000, random_state=SEED)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    aucs = cross_val_score(clf, X, labels, cv=skf, scoring='roc_auc')
    judgment_aucs[l] = [aucs.mean(), aucs.std()]

# Find emergence layers
def first_above(arr, threshold):
    idx = np.where(arr >= threshold)[0]
    return int(idx[0]) if len(idx) > 0 else -1

content_emerge = first_above(content_aucs[:, 0], 0.95)
judgment_emerge = first_above(judgment_aucs[:, 0], 0.95)
gap = judgment_emerge - content_emerge if (content_emerge >= 0 and judgment_emerge >= 0) else -999

print(f'Content AUC >= 0.95: Layer {content_emerge}')
print(f'Judgment AUC >= 0.95: Layer {judgment_emerge}')
print(f'Perception-Judgment gap: {gap} layers')

# Load Exp 3 reference
exp3_content = exp3_data['probe_content'][:, 0] if 'probe_content' in exp3_data else None  # col 0 = mean AUC
exp3_judgment = exp3_data['probe_judgment'][:, 0] if 'probe_judgment' in exp3_data else None

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Part B: Aesthetic Emergence', fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(range(N_LAYERS), content_aucs[:, 0], 'b-o', ms=3, label='Content probing')
ax.plot(range(N_LAYERS), judgment_aucs[:, 0], 'orange', ls='-', marker='s', ms=3, label='Judgment probing')
ax.axhline(0.95, ls=':', c='gray', alpha=0.5, label='AUC=0.95')
if content_emerge >= 0:
    ax.axvline(content_emerge, ls='--', c='blue', alpha=0.5)
if judgment_emerge >= 0:
    ax.axvline(judgment_emerge, ls='--', c='orange', alpha=0.5)
ax.set_xlabel('Layer'); ax.set_ylabel('ROC-AUC (5-fold CV)')
ax.set_title('Emergence: Content vs Judgment (Matched Poems)')
ax.legend(fontsize=8); ax.set_ylim(0.3, 1.05)

ax = axes[1]
ax.plot(range(N_LAYERS), content_aucs[:, 0], 'b-o', ms=3, label='DirA: Content')
ax.plot(range(N_LAYERS), judgment_aucs[:, 0], color='orange', ls='-', marker='s', ms=3, label='DirA: Judgment')
if exp3_content is not None:
    ax.plot(range(min(N_LAYERS, len(exp3_content))), exp3_content[:N_LAYERS], 'b--', alpha=0.5, label='Exp 3: Content')
if exp3_judgment is not None:
    ax.plot(range(min(N_LAYERS, len(exp3_judgment))), exp3_judgment[:N_LAYERS], color='orange', ls='--', alpha=0.5, label='Exp 3: Judgment')
ax.axhline(0.95, ls=':', c='gray', alpha=0.5)
ax.set_xlabel('Layer'); ax.set_ylabel('ROC-AUC')
ax.set_title('Comparison with Exp 3 (Synthetic)')
ax.legend(fontsize=8); ax.set_ylim(0.3, 1.05)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'partB_emergence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n{"="*60}')
print(f'PART B SUMMARY: Aesthetic Emergence')
print(f'{"="*60}')
print(f'{"Metric":<35} {"Exp 3":>10} {"Exp 9":>10} {"Dir A":>10}')
print(f'{"-"*65}')
print(f'{"Content AUC >= 0.95":<35} {"L3":>10} {"L0":>10} {"L"+str(content_emerge):>10}')
print(f'{"Judgment AUC >= 0.95":<35} {"L18":>10} {"L0":>10} {"L"+str(judgment_emerge):>10}')
print(f'{"Gap":<35} {"15":>10} {"0":>10} {gap:>10}')
print(f'\nSaved partB_emergence.png')

Content AUC >= 0.95: Layer 0
Judgment AUC >= 0.95: Layer 3
Perception-Judgment gap: 3 layers

PART B SUMMARY: Aesthetic Emergence
Metric                                   Exp 3      Exp 9      Dir A
-----------------------------------------------------------------
Content AUC >= 0.95                         L3         L0         L0
Judgment AUC >= 0.95                       L18         L0         L3
Gap                                         15          0          3

Saved partB_emergence.png


---
## Part C: Aesthetic Neurons (cf. Exp 5)

Neuron selectivity, offline masking, and online ablation at the best-discriminating layer.

In [12]:
# §9 Neuron selectivity + online ablation
# Cohen's d per neuron at best_layer
h_acts = acts_high[:, best_layer, :]  # (50, 4096)
l_acts = acts_low[:, best_layer, :]
mean_h = h_acts.mean(axis=0)
mean_l = l_acts.mean(axis=0)
std_h = h_acts.std(axis=0)
std_l = l_acts.std(axis=0)
pooled_std = np.sqrt((std_h**2 + std_l**2) / 2) + 1e-10
selectivity = np.abs(mean_h - mean_l) / pooled_std

# Contribution: |dir_weight| * |mean_diff|
contribution = np.abs(aesthetic_dir_norm[best_layer]) * np.abs(mean_h - mean_l)

# Top neurons
sorted_neurons = np.argsort(contribution)[::-1]
top_neurons = sorted_neurons[:100]

# Overlap with Exp 5
exp5_selectivity = exp5_data['selectivity'][16] if 'selectivity' in exp5_data else None  # Exp5 best=L16
exp5_contribution = exp5_data['contribution'][16] if 'contribution' in exp5_data else None

if exp5_contribution is not None:
    exp5_top100 = np.argsort(exp5_contribution)[::-1][:100]
    overlap = len(set(top_neurons.tolist()) & set(exp5_top100.tolist()))
else:
    overlap = -1

print(f'Selectivity at L{best_layer}: mean={selectivity.mean():.3f}, max={selectivity.max():.3f}')
print(f'Top-10 neurons: {sorted_neurons[:10]}')
print(f'Top-100 overlap with Exp 5: {overlap}/100')

# --- Online Ablation ---
STEER_PROMPTS = [
    'Describe a garden',
    'Write about the ocean at sunset',
    'Describe a quiet morning in the countryside',
    'Tell me about a beautiful piece of music',
    'Describe an ancient temple at dawn',
]

def measure_aesthetic_density(engine, prompts, layer=None, mask=None, max_tokens=128):
    densities = []
    for prompt in prompts:
        msgs = [{'role': 'user', 'content': prompt}]
        templated = engine.apply_chat_template(msgs) + NOTHINK_SUFFIX
        engine.reset()
        engine.clear_interventions()
        if mask is not None and layer is not None:
            engine.apply_mask(layer, mask)
        try:
            text = engine.generate(templated, max_tokens=max_tokens, temperature=0.0)
        except (UnicodeDecodeError, RuntimeError):
            text = ''
        engine.clear_interventions()
        words = text.lower().split()
        aes_count = sum(1 for w in words if w.strip('.,!?;:') in AESTHETIC_WORDS)
        densities.append(aes_count / max(len(words), 1))
    return np.mean(densities)

print('\n=== Online Ablation ===')
baseline_density = measure_aesthetic_density(engine, STEER_PROMPTS)
print(f'Baseline density: {baseline_density:.4f}')

# Group ablation sweep
print('\n--- Group ablation sweep ---')
group_sizes = [5, 10, 20, 50, 100, 200, 500, 1000]
group_results = []
for N in group_sizes:
    mask = np.ones(N_EMBD, dtype=np.float32)
    mask[sorted_neurons[:N]] = 0.0
    d = measure_aesthetic_density(engine, STEER_PROMPTS, layer=best_layer, mask=mask)
    change = (d - baseline_density) / max(baseline_density, 1e-6) * 100
    group_results.append(d)
    print(f'  N={N:>5}: density={d:.4f} ({"+" if change>0 else ""}{change:.1f}%)')

# Random control
print('\n--- Random control ---')
random_results = []
for N in group_sizes:
    mask = np.ones(N_EMBD, dtype=np.float32)
    rand_neurons = np.random.RandomState(SEED+1).choice(N_EMBD, N, replace=False)
    mask[rand_neurons] = 0.0
    d = measure_aesthetic_density(engine, STEER_PROMPTS, layer=best_layer, mask=mask)
    random_results.append(d)

# Layer sensitivity (top-20 at each layer)
print('\n--- Layer sensitivity ---')
test_layers = [0, 3, 8, 12, 16, 20, 24, 28, 32, 35]
layer_sens = []
for test_l in test_layers:
    h_l = acts_high[:, test_l, :]
    l_l = acts_low[:, test_l, :]
    sel_l = np.abs(h_l.mean(0) - l_l.mean(0)) / (np.sqrt((h_l.std(0)**2 + l_l.std(0)**2)/2) + 1e-10)
    contrib_l = np.abs(aesthetic_dir_norm[test_l]) * np.abs(h_l.mean(0) - l_l.mean(0))
    top20 = np.argsort(contrib_l)[::-1][:20]
    mask = np.ones(N_EMBD, dtype=np.float32)
    mask[top20] = 0.0
    d = measure_aesthetic_density(engine, STEER_PROMPTS, layer=test_l, mask=mask)
    change = (d - baseline_density) / max(baseline_density, 1e-6) * 100
    layer_sens.append((test_l, d, change))
    print(f'  L{test_l:>2}: density={d:.4f} ({"+" if change>0 else ""}{change:.1f}%)')

engine.clear_interventions()

Selectivity at L4: mean=0.336, max=1.763
Top-10 neurons: [1838 2579 4081 3470 1231 4050 3673 2284 2170  822]
Top-100 overlap with Exp 5: 10/100

=== Online Ablation ===
Baseline density: 0.0171

--- Group ablation sweep ---
  N=    5: density=0.0166 (-2.7%)
  N=   10: density=0.0167 (-2.2%)
  N=   20: density=0.0236 (+38.1%)
  N=   50: density=0.0215 (+25.9%)
  N=  100: density=0.0227 (+32.7%)
  N=  200: density=0.0204 (+19.5%)
  N=  500: density=0.0222 (+29.9%)
  N= 1000: density=0.0132 (-22.6%)

--- Random control ---

--- Layer sensitivity ---
  L 0: density=0.0000 (-100.0%)
  L 3: density=0.0164 (-4.1%)
  L 8: density=0.0146 (-14.5%)
  L12: density=0.0091 (-46.9%)
  L16: density=0.0265 (+55.3%)
  L20: density=0.0198 (+15.9%)
  L24: density=0.0165 (-3.3%)
  L28: density=0.0222 (+29.9%)
  L32: density=0.0179 (+5.1%)
  L35: density=0.0215 (+26.3%)


In [13]:
# §10 Part C Visualization + Summary
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Part C: Aesthetic Neurons (L{best_layer})', fontsize=14, fontweight='bold')

# 1. Selectivity distribution
ax = axes[0, 0]
ax.hist(selectivity, bins=50, color='steelblue', alpha=0.7)
ax.axvline(np.percentile(selectivity, 95), ls='--', c='red', label='95th pctile')
ax.set_xlabel("Cohen's d"); ax.set_ylabel('Count')
ax.set_title(f'Neuron Selectivity (L{best_layer})'); ax.legend()

# 2. Group ablation
ax = axes[0, 1]
ax.plot(group_sizes, group_results, 'r-o', ms=5, label='Targeted')
ax.plot(group_sizes, random_results, 'gray', ls='--', marker='s', ms=4, label='Random')
ax.axhline(baseline_density, ls=':', c='green', label=f'Baseline={baseline_density:.3f}')
ax.set_xscale('log'); ax.set_xlabel('N Neurons Ablated')
ax.set_ylabel('Aesthetic Density'); ax.set_title('Group Ablation Sweep')
ax.legend(fontsize=8)

# 3. Layer sensitivity
ax = axes[1, 0]
changes = [ls[2] for ls in layer_sens]
colors = ['green' if c > 0 else 'red' for c in changes]
ax.bar([f'L{ls[0]}' for ls in layer_sens], changes, color=colors)
ax.axhline(0, ls='-', c='black', lw=0.5)
ax.set_ylabel('Density Change (%)'); ax.set_title('Layer Sensitivity (Top-20 Ablated)')

# 4. Contribution comparison with Exp 5
ax = axes[1, 1]
if exp5_contribution is not None:
    ax.scatter(exp5_contribution, contribution, s=3, alpha=0.3, c='purple')
    from scipy.stats import pearsonr
    r_val = pearsonr(exp5_contribution, contribution)[0]
    ax.set_xlabel('Exp 5 Contribution'); ax.set_ylabel('Dir A Contribution')
    ax.set_title(f'Contribution: Exp 5 vs Dir A (r={r_val:.3f})')
else:
    ax.text(0.5, 0.5, 'Exp 5 data not available', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'partC_neurons.png', dpi=150, bbox_inches='tight')
plt.show()

# Identify brake/driver layers
brake_layers = [ls[0] for ls in layer_sens if ls[2] > 10]
driver_layers = [ls[0] for ls in layer_sens if ls[2] < -10]

print(f'\n{"="*60}')
print(f'PART C SUMMARY: Aesthetic Neurons')
print(f'{"="*60}')
print(f'{"Metric":<35} {"Exp 5":>12} {"Exp 9":>12} {"Dir A":>12}')
print(f'{"-"*71}')
print(f'{"Baseline density":<35} {"0.95%":>12} {"2.48%":>12} {baseline_density*100:>11.2f}%')
print(f'{"Top-100 overlap w/ Exp 5":<35} {"—":>12} {"19/100":>12} {overlap:>10}/100')
print(f'{"Brake layers (>+10%)":<35} {"L8-L24":>12} {"L20":>12} {str(brake_layers):>12}')
print(f'{"Driver layers (<-10%)":<35} {"L32-35":>12} {"L3,24,28,32":>12} {str(driver_layers):>12}')
print(f'\nSaved partC_neurons.png')


PART C SUMMARY: Aesthetic Neurons
Metric                                     Exp 5        Exp 9        Dir A
-----------------------------------------------------------------------
Baseline density                           0.95%        2.48%        1.71%
Top-100 overlap w/ Exp 5                       —       19/100         10/100
Brake layers (>+10%)                      L8-L24          L20 [16, 20, 28, 35]
Driver layers (<-10%)                     L32-35  L3,24,28,32   [0, 8, 12]

Saved partC_neurons.png


---
## Final: Save & Summary

In [14]:
# §11 Save all results + comprehensive comparison
np.savez(OUTPUT_DIR / 'dirA_results.npz',
    aesthetic_dir=aesthetic_dir, aesthetic_dir_norm=aesthetic_dir_norm,
    layer_aucs=layer_aucs, judgment_aucs=judgment_aucs,
    cos_exp2=cos_exp2, cos_exp9=cos_exp9,
    best_layer=best_layer, selectivity=selectivity, contribution=contribution,
    content_emerge=content_emerge, judgment_emerge=judgment_emerge, gap=gap,
    high_scores=high_scores, low_scores=low_scores,
    group_results=np.array(group_results), random_results=np.array(random_results),
    group_sizes=np.array(group_sizes), layer_sens=np.array(layer_sens, dtype=object),
    brake_layers=np.array(brake_layers), driver_layers=np.array(driver_layers))

print(f'Results saved to {OUTPUT_DIR / "dirA_results.npz"}')

print(f'\n{"="*70}')
print(f' DIRECTION A — COMPREHENSIVE COMPARISON')
print(f'{"="*70}')
print(f'\n--- PART A: Subspace ---')
print(f'  Best layer: Exp 2=L16, Exp 9=L0, Dir A=L{best_layer}')
print(f'  Peak AUC:   Exp 2=1.000, Exp 9=1.000, Dir A={best_auc:.4f}')
print(f'  Cosine vs Exp 2 @ L16: Exp 9=0.310, Dir A={cos_exp2[16]:.3f}')
print(f'  Cosine vs Exp 9 @ L0:  Dir A={cos_exp9[0]:.3f}')
print(f'\n--- PART B: Emergence ---')
print(f'  Content >= 0.95: Exp 3=L3, Exp 9=L0, Dir A=L{content_emerge}')
print(f'  Judgment >= 0.95: Exp 3=L18, Exp 9=L0, Dir A=L{judgment_emerge}')
print(f'  Gap: Exp 3=15, Exp 9=0, Dir A={gap}')
print(f'\n--- PART C: Neurons ---')
print(f'  Brake layers: Exp 5=L8-24, Dir A={brake_layers}')
print(f'  Driver layers: Exp 5=L32-35, Dir A={driver_layers}')
print(f'\nAll plots saved to: {OUTPUT_DIR}')
print(f'{"="*70}')

Results saved to /workspace/Data/direction_A/dirA_results.npz

 DIRECTION A — COMPREHENSIVE COMPARISON

--- PART A: Subspace ---
  Best layer: Exp 2=L16, Exp 9=L0, Dir A=L4
  Peak AUC:   Exp 2=1.000, Exp 9=1.000, Dir A=0.9680
  Cosine vs Exp 2 @ L16: Exp 9=0.310, Dir A=0.166
  Cosine vs Exp 9 @ L0:  Dir A=-0.265

--- PART B: Emergence ---
  Content >= 0.95: Exp 3=L3, Exp 9=L0, Dir A=L0
  Judgment >= 0.95: Exp 3=L18, Exp 9=L0, Dir A=L3
  Gap: Exp 3=15, Exp 9=0, Dir A=3

--- PART C: Neurons ---
  Brake layers: Exp 5=L8-24, Dir A=[16, 20, 28, 35]
  Driver layers: Exp 5=L32-35, Dir A=[0, 8, 12]

All plots saved to: /workspace/Data/direction_A
